# RFM 用户分群综合案例

本篇对应 `08_综合案例_RFM案例`，使用 Pandas 手动实现 RFM 会员价值度模型。案例覆盖 Excel 多工作表读取、数据清洗、时间差计算、分组聚合、分箱打分、用户分群、3D 可视化以及结果导出。

> 说明：所有代码放在 Markdown 代码块中，运行时需要准备课程提供的 `data/sales.xlsx`，并按本机环境安装 `pandas`、`numpy` 和可选的 `pyecharts`。

## 一、RFM 模型是什么

RFM 是根据会员交易行为评估用户价值的经典模型：

- **R（Recency）最近一次购买时间间隔**：距离分析截止时间越近越好，因此间隔越小，R 得分越高；
- **F（Frequency）购买频率**：在统计周期内购买次数越多越好；
- **M（Monetary）购买金额**：在统计周期内累计消费金额越高越好。

RFM 必须基于一个明确的时间点计算。分析截止日期变化后，用户的最近购买间隔、频率和金额可能发生变化，因此结果不能脱离时间范围解读。

## 二、从 RFM 得分到用户分群

课程将 R、F、M 分别离散为 1、2、3 三个等级，最多形成 `3 × 3 × 3 = 27` 个分组。把三个分数组合成字符串，例如 `333` 表示 R、F、M 都处于高等级。

| R | F | M | 用户类别 |
| --- | --- | --- | --- |
| 高 | 高 | 高 | 重要价值用户 |
| 高 | 低 | 高 | 重要发展用户 |
| 低 | 高 | 高 | 重要保持用户 |
| 低 | 低 | 高 | 重要挽留用户 |
| 高 | 高 | 低 | 一般价值用户 |
| 高 | 低 | 低 | 一般发展用户 |
| 低 | 高 | 低 | 一般保持用户 |
| 低 | 低 | 低 | 一般挽留用户 |

“高/低”是相对于本次业务区间和打分规则而言，不能直接等同于绝对好坏。

## 三、案例目标与数据说明

### 3.1 业务目标

1. 对用户做可解释的价值分组；
2. 概括各组的用户特征，为精细化运营提供依据；
3. 将结果导出为 Excel，供业务部门二次加工；
4. 同时写入 MySQL，作为其他模型的输入特征。

### 3.2 数据集

案例文件为 `data/sales.xlsx`，包含 5 个工作表：`2015`、`2016`、`2017`、`2018` 和 `会员等级`。前四张表记录订单，字段包括：

- `会员ID`：用户唯一标识；
- `订单号`：订单唯一标识；
- `提交日期`：订单提交时间；
- `订单金额`：订单金额。

会员等级表包含 `会员ID` 和 `会员等级`，可在后续通过会员 ID 进行关联。课程数据中四年的订单量分别约为 30,774、41,278、50,839 和 81,349 行，包含少量缺失值和异常金额。

## 四、读取多工作表数据

`pd.read_excel` 传入工作表名称列表时，会返回一个字典：键是工作表名，值是对应的 DataFrame。

```python
import time
import numpy as np
import pandas as pd

sheet_names = ["2015", "2016", "2017", "2018", "会员等级"]
sheet_datas = pd.read_excel(
    "data/sales.xlsx",
    sheet_name=sheet_names,
)

# 查看 2015 年订单
sheet_datas["2015"].head()
```

读取后先检查每张表的结构、数据类型、缺失值和数值分布：

```python
for name in sheet_names:
    print(f"===== {name} =====")
    print(sheet_datas[name].head())
    print(sheet_datas[name].info())
    print(sheet_datas[name].describe())
```

## 五、数据预处理

### 5.1 处理缺失值和异常订单

课程与业务确认后采用以下规则：删除缺失记录，删除订单金额小于等于 1 的订单。金额极大的订单保留，因为它可能是一次性购买多个大件商品；低于 1 元的订单通常是优惠券或非正常支付记录。

```python
# 只处理前四年的订单表，不处理会员等级表
for name in sheet_names[:-1]:
    sheet_datas[name] = sheet_datas[name].dropna()
    sheet_datas[name] = sheet_datas[name][
        sheet_datas[name]["订单金额"] > 1
    ]

    # 每年最后一笔订单日期，作为该年计算参照点
    sheet_datas[name]["max_year_date"] = (
        sheet_datas[name]["提交日期"].max()
    )
```

清洗后再次检查：

```python
for name in sheet_names[:-1]:
    print(name, sheet_datas[name].isnull().sum())
    print(sheet_datas[name]["订单金额"].describe())
```

### 5.2 合并四年订单

每个年度表的字段结构一致，使用 `pd.concat` 沿行合并。

```python
yearly_tables = [sheet_datas[name] for name in sheet_names[:-1]]
data_merge = pd.concat(yearly_tables, ignore_index=True)

# 从订单日期提取年份
data_merge["year"] = data_merge["提交日期"].dt.year

# 订单距离该年度最后一天的间隔
data_merge["date_interval"] = (
    data_merge["max_year_date"] - data_merge["提交日期"]
).dt.days
```

`dt` 是 Pandas 时间序列的访问器。常见属性包括 `year`、`month`、`dayofweek`、`days`、`quarter` 等。时间差转换为天数后，才能方便地进行 R 分箱。

## 六、计算 R、F、M 原始值

以 `year` 和 `会员ID` 为联合分组键：

- R：取 `date_interval` 的最小值，即该用户当年最近一次购买距离年末的最短间隔；
- F：统计订单号数量；
- M：对订单金额求和。

```python
rfm_gb = (
    data_merge
    .groupby(["year", "会员ID"], as_index=False)
    .agg({
        "date_interval": "min",
        "订单号": "count",
        "订单金额": "sum",
    })
)

rfm_gb.columns = ["year", "会员ID", "r", "f", "m"]
rfm_gb.head()
```

`as_index=False` 让分组键继续作为普通列，便于后续写出文件、绘图和连接其他表。

### 6.1 查看分布，决定分箱边界

分箱前不能盲目使用等距区间，应先观察分布并结合业务经验。

```python
rfm_gb[["r", "f", "m"]].describe().T
```

课程案例观察到：

- R 和 M 的分布相对分散，可以参考四分位数选择边界；
- F 大量集中在 1 次，单纯按四分位数难以区分用户；
- F 的边界结合业务经验设置为 2 次和 5 次，分别代表复购和高频购买。

本案例采用三段区间：

```python
r_bins = [-1, 79, 255, 365]
f_bins = [0, 2, 5, 130]
m_bins = [0, 69, 1199, 206252]
```

边界的最小值要小于实际最小值，最大值要覆盖实际最大值。`pd.cut` 默认采用左开右闭区间，因此边界值归入右侧区间。

## 七、使用 `pd.cut` 分箱打分

R、F、M 的“好方向”不同：R 的间隔越小越好，所以 R 的标签要反向；F 和 M 的值越大越好，标签按升序增加。

```python
rfm_gb["r_label"] = pd.cut(
    rfm_gb["r"],
    bins=r_bins,
    labels=[3, 2, 1],
    include_lowest=True,
)
rfm_gb["f_label"] = pd.cut(
    rfm_gb["f"],
    bins=f_bins,
    labels=[1, 2, 3],
    include_lowest=True,
)
rfm_gb["m_label"] = pd.cut(
    rfm_gb["m"],
    bins=m_bins,
    labels=[1, 2, 3],
    include_lowest=True,
)
```

检查分箱是否覆盖了全部数据：

```python
for column in ["r_label", "f_label", "m_label"]:
    print(column, rfm_gb[column].isna().sum())
```

如果出现大量 `NaN`，应检查边界是否覆盖最小值和最大值，以及数据中是否存在异常类型。

## 八、组合 RFM 分组编码

将三个标签转换为字符串后拼接，得到便于分组和展示的编码。

```python
for column in ["r_label", "f_label", "m_label"]:
    rfm_gb[column] = rfm_gb[column].astype(str)

rfm_gb["rfm_group"] = (
    rfm_gb["r_label"]
    + rfm_gb["f_label"]
    + rfm_gb["m_label"]
)

# 统计每年每种分组的用户数
group_counts = (
    rfm_gb
    .groupby(["year", "rfm_group"], as_index=False)["会员ID"]
    .count()
    .rename(columns={"会员ID": "number"})
)
```

也可以使用 `Series.str.cat` 拼接字符串：

```python
left = rfm_gb["r_label"].str.cat(rfm_gb["f_label"])
rfm_gb["rfm_group"] = left.str.cat(rfm_gb["m_label"])
```

## 九、用 Pyecharts 绘制 RFM 3D 柱形图

为了观察不同年份、不同 RFM 分组的人数变化，课程使用 Pyecharts 的 `Bar3D`：

- x 轴：RFM 分组；
- y 轴：年份；
- z 轴：用户数量。

```python
from pyecharts.charts import Bar3D
import pyecharts.options as opts

# 将分组结果整理为 3D 图需要的三列
display_data = (
    rfm_gb
    .groupby(["rfm_group", "year"], as_index=False)["会员ID"]
    .count()
    .rename(columns={"会员ID": "number"})
)
display_data["rfm_group"] = display_data["rfm_group"].astype(int)

range_color = [
    "#313695", "#4575b4", "#74add1", "#abd9e9",
    "#e0f3f8", "#ffffbf", "#fee090", "#fdae61",
    "#f46d43", "#d73027", "#a50026",
]

bar3d = (
    Bar3D()
    .add(
        "",
        [row.tolist() for row in display_data.values],
        xaxis3d_opts=opts.Axis3DOpts(
            type_="category", name="RFM分组"
        ),
        yaxis3d_opts=opts.Axis3DOpts(
            type_="category", name="年份"
        ),
        zaxis3d_opts=opts.Axis3DOpts(
            type_="value", name="会员数量"
        ),
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="RFM分组结果"),
        visualmap_opts=opts.VisualMapOpts(
            max_=int(display_data["number"].max()),
            range_color=range_color,
        ),
    )
)

bar3d.render("rfm_group.html")
```

在 Jupyter 中也可以尝试 `bar3d.render_notebook()`。交互式图表可以旋转和缩放，便于观察特定分组在不同年份的变化。

## 十、结果解读与运营思路

RFM 分组不是终点，关键是把分组特征转成运营动作。课程案例强调：

- 大体量用户群适合通过系统、活动和批量触达运营；
- 中等规模用户群可以结合产品运营和人工运营；
- 数量少但 R、F、M 都高的用户，应提供 VIP、专享服务或绿色通道；
- 最近购买间隔较长的用户，需要优先考虑召回和复购刺激；
- 购买频率高但金额偏低的用户，可以通过关联销售、组合商品和向上销售提升客单价；
- 最近购买很近但频率不高的用户，可以通过新品推荐、节日活动和持续触达提高复购。

具体策略必须结合行业、商品购买周期、营销资源和业务目标确认，不能只凭 RFM 编码自动决定。

## 十一、保存分析结果

### 11.1 导出 Excel

```python
rfm_gb.to_excel(
    "sales_rfm_score.xlsx",
    index=False,
)
```

### 11.2 写入 MySQL

```python
from sqlalchemy import create_engine

# 建议从环境变量读取账号和密码
engine = create_engine(
    "mysql+pymysql://用户名:密码@localhost:3306/rfm_db?charset=utf8"
)

rfm_gb.to_sql(
    "sales_rfm_score",
    engine,
    index=False,
    if_exists="append",
)
```

写入后应检查 DataFrame 行数与数据库表行数是否一致，并确认表字段类型、主键和重复写入策略。

## 十二、案例中的关键检查点

### 时间口径

R 是相对某个截止时间点计算的间隔。按年度分析时，课程将每一年的最后订单日期作为当年参照点；如果改成统一截止日，R 的业务解释也要同步修改。

### 异常值规则

订单金额小于 1 的记录在本案例中被视为无实际分析意义并删除。极大金额订单是否异常，需要和业务方确认，不能简单按最大值删除。

### 分箱边界

`pd.cut` 的区间默认是左开右闭，最小边界应小于真实最小值。分箱数量越多，用户群越细，但运营策略数量也会增加。

### 数据类型

日期列应先转换为 datetime，标签列在拼接前转换为字符串，数据库导出前确认数值列和日期列的类型。

### 结果完整性

在导出前检查分箱缺失、分组数量、重复用户和年度覆盖范围，避免把清洗或分箱错误带入业务结果。

## 十三、RFM 模型的局限与扩展

- RFM 更适合存在重复购买行为的零售、会员和电商场景；购买周期很长的行业需要提高 R 和 M 的权重关注度。
- 不同阶段可以使用 2 段或 3 段分箱。2 段最多 8 组，策略更简单；3 段最多 27 组，区分更细但运营成本更高。
- R、F、M 的边界既可以参考分位数，也可以采用业务阈值、平均值、中位数或二八法则。
- 如果需要综合权重，可以使用专家打分或 AHP 等方法，但应记录权重依据。
- RFM 结果可以作为其他数据挖掘模型的特征，但需要继续验证数据质量、时间泄漏和业务稳定性。

## 十四、复习版完整流程

```python
# 1. 读取 Excel 多工作表
sheet_names = ["2015", "2016", "2017", "2018", "会员等级"]
sheets = pd.read_excel("data/sales.xlsx", sheet_name=sheet_names)

# 2. 清洗四年订单并添加年度参照日期
for name in sheet_names[:-1]:
    table = sheets[name].dropna()
    table = table[table["订单金额"] > 1].copy()
    table["max_year_date"] = table["提交日期"].max()
    sheets[name] = table

# 3. 合并订单并计算日期间隔
orders = pd.concat(
    [sheets[name] for name in sheet_names[:-1]],
    ignore_index=True,
)
orders["year"] = orders["提交日期"].dt.year
orders["date_interval"] = (
    orders["max_year_date"] - orders["提交日期"]
).dt.days

# 4. 按年度和会员计算 R/F/M
rfm = (
    orders.groupby(["year", "会员ID"], as_index=False)
    .agg(r=("date_interval", "min"),
         f=("订单号", "count"),
         m=("订单金额", "sum"))
)

# 5. 分箱、打分、组合分组
rfm["r_score"] = pd.cut(rfm["r"], [-1, 79, 255, 365], labels=[3, 2, 1], include_lowest=True)
rfm["f_score"] = pd.cut(rfm["f"], [0, 2, 5, 130], labels=[1, 2, 3], include_lowest=True)
rfm["m_score"] = pd.cut(rfm["m"], [0, 69, 1199, 206252], labels=[1, 2, 3], include_lowest=True)
rfm["rfm_group"] = (
    rfm["r_score"].astype(str)
    + rfm["f_score"].astype(str)
    + rfm["m_score"].astype(str)
)
```

## 本篇小结

RFM 案例把前面学到的 Excel 读取、缺失值处理、时间运算、`concat`、`groupby`、`agg`、`pd.cut`、可视化和结果导出串成了一条完整的数据分析链路。最需要反复确认的是时间口径、异常金额规则、R/F/M 的评分方向和分箱边界；这些业务决策决定了最终分群是否有解释力。